# GTSRB-Training auf Google Colab

Dieses Notebook trainiert das ResNet-18 aus diesem Repository auf einer kostenlosen Colab-GPU, misst es auf dem offiziellen GTSRB-Testset und exportiert es als ONNX-Modell für die Live-Demo unter Windows.

**Vorher:** *Laufzeit → Laufzeittyp ändern → T4 GPU* wählen. Dann alle Zellen der Reihe nach ausführen.

Checkpoints und Ergebnisse werden, wenn erlaubt, in Google Drive gespeichert. Bricht die Colab-Sitzung ab, ist das beste Modell dann nicht verloren.

## 1. GPU prüfen

In [ ]:
import torch
print('PyTorch', torch.__version__)
assert torch.cuda.is_available(), 'Keine GPU! Laufzeit -> Laufzeittyp ändern -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Code holen

Trage die URL deines Repositorys ein (GitHub oder das GitLab-Repo des Kurses). Die lokalen Änderungen müssen vorher gepusht sein.
Bei einem privaten Repo: `https://<user>:<token>@github.com/...` verwenden (Token nicht im Notebook committen).

Alternative ohne Git: Projektordner als `.zip` hochladen (Ordnersymbol links) und die zweite Zelle nutzen.

In [ ]:
REPO_URL = 'https://github.com/tuananh-aa-2001/Traffic-Sign-Recognition-System.git'  # anpassen
BRANCH = 'pipeline-fixes-colab'  # nach dem Merge in master: 'master'

import os
%cd /content
if not os.path.exists('/content/project'):
    !git clone --depth 1 -b {BRANCH} {REPO_URL} project
%cd /content/project
!git log --oneline -3

In [ ]:
# Alternative: hochgeladene ZIP entpacken (nur ausführen, wenn du nicht klonst)
# !unzip -q /content/projekt.zip -d /content/project_zip
# %cd /content/project_zip/<Ordnername>

## 3. Abhängigkeiten installieren

In [ ]:
# PyTorch, OpenCV, NumPy usw. sind in Colab vorinstalliert, pip ergänzt nur das Fehlende
!pip install -q -r requirements.txt

## 4. Speicherort wählen und Colab-Konfiguration schreiben

Colab fragt nach Zugriff auf Google Drive. Erlaubst du ihn, landen Checkpoints in `MyDrive/dlacv_gtsrb`. Lehnst du ab, wird lokal gespeichert; dann Ergebnisse am Ende unbedingt herunterladen.

In [ ]:
import yaml

# Ergebnisse möglichst in Google Drive speichern (überlebt einen Sitzungsabbruch).
# Ohne Drive-Freigabe wird lokal gespeichert und am Ende als ZIP heruntergeladen.
DRIVE_DIR = '/content/outputs'
try:
    from google.colab import drive
    drive.mount('/content/drive', timeout_ms=300000)
    DRIVE_DIR = '/content/drive/MyDrive/dlacv_gtsrb'
except Exception as e:
    print('Google Drive nicht verbunden, speichere lokal unter', DRIVE_DIR, '-', e)
os.makedirs(DRIVE_DIR, exist_ok=True)

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['training']['num_workers'] = os.cpu_count()              # Colab hat meist nur 2 CPU-Kerne
cfg['checkpoint']['save_dir'] = f'{DRIVE_DIR}/checkpoints'
cfg['logging']['tensorboard_dir'] = f'{DRIVE_DIR}/tensorboard'

with open('configs/colab.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(yaml.safe_dump(cfg['training']))

## 5. Datensatz laden

Lädt ca. 300 MB von der offiziellen GTSRB-Quelle und entpackt Trainingsbilder, Testbilder und Testlabels nach `data/raw/GTSRB/`.

In [ ]:
!python src/utils/download_dataset.py --data_dir data/raw

## 6. Training

Validierung = 10 % der GTSRB-*Tracks* (alle Frames eines Schildes landen entweder im Training oder in der Validierung).
Early Stopping beendet das Training, wenn sich die Validierungsgenauigkeit 10 Epochen lang nicht verbessert.
Die Laufzeit notierst du dir am besten für den Bericht.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DRIVE_DIR}/tensorboard

In [ ]:
!python train.py --config configs/colab.yaml 2>&1 | tee {DRIVE_DIR}/train_log.txt

## 7. Evaluation auf dem offiziellen Testset (12.630 Bilder)

In [ ]:
!python evaluate.py --config configs/colab.yaml --save_cm --benchmark 2>&1 | tee {DRIVE_DIR}/eval_log.txt

In [ ]:
from IPython.display import Image, display
display(Image('results/training_curves.png', width=900))
display(Image('results/confusion_matrix.png', width=900))

## 8. ONNX-Export für die Windows-Demo

In [ ]:
!python export_onnx.py --config configs/colab.yaml --output models/onnx/gtsrb_model.onnx --verify --benchmark

## 9. Ergebnisse sichern und herunterladen

Lege danach auf deinem Windows-Rechner ab:
- `gtsrb_model.onnx` nach `models/onnx/`
- `best_model.pth` nach `models/checkpoints/` (für Grad-CAM und Fine-Tuning)

In [ ]:
import shutil
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
shutil.copy('models/onnx/gtsrb_model.onnx', DRIVE_DIR)
for name in os.listdir('results'):
    src = os.path.join('results', name)
    if os.path.isfile(src):
        shutil.copy(src, f'{DRIVE_DIR}/results')

!cd {DRIVE_DIR} && zip -q -r /content/gtsrb_ergebnisse.zip gtsrb_model.onnx checkpoints/best_model.pth results train_log.txt eval_log.txt
from google.colab import files
files.download('/content/gtsrb_ergebnisse.zip')